# Model Training


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from pathlib import Path
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from imblearn.over_sampling import RandomOverSampler

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

DEFAULT_SCENARIO_CLASS_NAMES = [
    "Botnet",
    "Miner",
    "Trojan-Agent",
    "Cloud Attack",
    "Data-Caching",
    "Database",
    "Media-Streaming",
    "Web-Serving",
]

def load_label_names():
    for path in [Path("../config/label_mapping.json"), Path("config/label_mapping.json")]:
        if path.exists():
            with open(path) as f:
                cfg = json.load(f)
            return cfg.get("scenario_class_names", cfg.get("class_names", DEFAULT_SCENARIO_CLASS_NAMES))
    return DEFAULT_SCENARIO_CLASS_NAMES

SCENARIO_CLASS_NAMES = load_label_names()
CLASS_NAMES = SCENARIO_CLASS_NAMES
METADATA_COLS = ["session_id", "pod_name", "namespace", "ngram_index", "flow_window"]
TARGET_COLS = ["label", "scenario_label"]

## Load Dataset

In [ ]:
syscall_df = pd.read_csv("../feature_engineering/dataset/syscall_dataset.csv")
network_df = pd.read_csv("../feature_engineering/dataset/network_flow_dataset.csv")

def load_label_rules():
    for path in [Path("../config/label_mapping.json"), Path("config/label_mapping.json")]:
        if path.exists():
            with open(path) as f:
                cfg = json.load(f)
            return sorted(cfg["rules"], key=lambda rule: len(rule["pattern"]), reverse=True)
    return []

LABEL_RULES = load_label_rules()

def normalize_name(value):
    return (value or "").lower().replace("_", "-")

def infer_label_from_pod(pod_name):
    pod_lower = normalize_name(pod_name)
    for rule in LABEL_RULES:
        if normalize_name(rule["pattern"]) in pod_lower:
            return int(rule.get("scenario_label", rule.get("label")))
    return -1

def ensure_scenario_target(df):
    df = df.copy()

    if "scenario_label" not in df.columns:
        if "pod_name" in df.columns and LABEL_RULES:
            df["scenario_label"] = df["pod_name"].apply(infer_label_from_pod)
            unknown = df[df["scenario_label"] < 0]
            if not unknown.empty:
                print(f"Warning: dropping {len(unknown):,} rows with unknown pod labels")
                print(unknown["pod_name"].value_counts().head(15))
                df = df[df["scenario_label"] >= 0].copy()
        else:
            df["scenario_label"] = df["label"]

    df["label"] = df["scenario_label"]
    return df

syscall_df = ensure_scenario_target(syscall_df)
network_df = ensure_scenario_target(network_df)

print(f"Syscall dataset shape: {syscall_df.shape}")
print(f"Network dataset shape: {network_df.shape}")

print("\nSyscall sample:")
display(syscall_df.head())
print("\nNetwork sample:")
display(network_df.head())

## Exploratory Data Analysis (EDA)

In [ ]:
# Info dataset
print("="*50)
print("SYSCALL DATA INFO")
print("="*50)
print(syscall_df.info())

print("\n" + "="*50)
print("NETWORK DATA INFO")
print("="*50)
print(network_df.info())

In [ ]:
# Missing values
print("\n" + "="*50)
print("MISSING VALUES - SYSCALL")
print("="*50)
print(syscall_df.isnull().sum())

print("\n" + "="*50)
print("MISSING VALUES - NETWORK")
print("="*50)
print(network_df.isnull().sum())

In [ ]:
print("\n" + "="*50)
print("DISTRIBUSI TARGET")
print("="*50)
print("Syscall scenario:", syscall_df["scenario_label"].value_counts().sort_index().to_dict())
print("Network scenario:", network_df["scenario_label"].value_counts().sort_index().to_dict())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
syscall_df["scenario_label"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_title("Syscall - Scenario Label")
network_df["scenario_label"].value_counts().sort_index().plot(kind="bar", ax=axes[1])
axes[1].set_title("Network - Scenario Label")
for ax in axes:
    ax.set_xlabel("Label")
    ax.set_ylabel("Rows")
plt.tight_layout()
plt.show()


## Data Preparation

In [ ]:
def prepare_xy(df, target_col):
    drop_cols = [col for col in METADATA_COLS + TARGET_COLS if col in df.columns]
    X = df.drop(columns=drop_cols, errors="ignore")
    X = X.select_dtypes(include=[np.number]).fillna(0)
    y = df[target_col].astype(int)
    return X, y

X_syscall, y_syscall = prepare_xy(syscall_df, "scenario_label")
X_network, y_network = prepare_xy(network_df, "scenario_label")

print(f"Syscall features: {X_syscall.shape}")
print(f"Network features: {X_network.shape}")
print(f"Syscall feature columns: {X_syscall.columns.tolist()}")
print(f"Network feature columns: {X_network.columns.tolist()}")


In [ ]:
def split_dataset(df, X, y, name):
    if "session_id" in df.columns and df["session_id"].nunique() > 1:
        group_col = "session_id"
    elif "pod_name" in df.columns and df["pod_name"].nunique() > 1:
        group_col = "pod_name"
    else:
        group_col = None

    if group_col:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
        train_idx, test_idx = next(splitter.split(X, y, groups=df[group_col]))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )

    print(f"{name} split group: {group_col or 'stratified row split'}")
    print(f"{name} - Train: {X_train.shape}, Test: {X_test.shape}")
    print(f"{name} train labels: {Counter(y_train)}")
    print(f"{name} test labels : {Counter(y_test)}")
    return X_train, X_test, y_train, y_test

X_train_sys, X_test_sys, y_train_sys, y_test_sys = split_dataset(
    syscall_df, X_syscall, y_syscall, "Syscall Scenario"
)
X_train_net, X_test_net, y_train_net, y_test_net = split_dataset(
    network_df, X_network, y_network, "Network Scenario"
)


## Class Balancing

In [ ]:
def balance_training_data(X_train, y_train, name, max_growth=5, max_majority_ratio=0.5):
    counts = pd.Series(y_train).value_counts().sort_index()
    print(f"{name} - Before balancing: {counts.to_dict()}")

    if len(counts) < 2:
        print(f"{name} - Skipped: only one class present.")
        return X_train, y_train

    majority = int(counts.max())
    ratio_cap = max(1, int(majority * max_majority_ratio))
    sampling_strategy = {}
    for label, count in counts.items():
        count = int(count)
        target = min(majority, ratio_cap, count * max_growth)
        sampling_strategy[int(label)] = max(count, target)

    sampler = RandomOverSampler(sampling_strategy=sampling_strategy, random_state=42)
    X_balanced, y_balanced = sampler.fit_resample(X_train, y_train)
    print(f"{name} - Sampling strategy: {sampling_strategy}")
    print(f"{name} - After balancing: {pd.Series(y_balanced).value_counts().sort_index().to_dict()}")
    return X_balanced, y_balanced

X_train_sys_balanced, y_train_sys_balanced = balance_training_data(X_train_sys, y_train_sys, "Syscall Scenario")
X_train_net_balanced, y_train_net_balanced = balance_training_data(X_train_net, y_train_net, "Network Scenario")


## Train Decision Tree Model

In [ ]:
def make_dt():
    return DecisionTreeClassifier(
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        criterion="gini",
        class_weight=None,
        random_state=42,
    )

dt_syscall = make_dt()
dt_network = make_dt()

print("Training scenario syscall model...")
dt_syscall.fit(X_train_sys_balanced, y_train_sys_balanced)
print("Training scenario network model...")
dt_network.fit(X_train_net_balanced, y_train_net_balanced)
print("Scenario models trained")


## Model Evaluation

In [ ]:
def class_names_for_labels(labels, class_names):
    labels = sorted(pd.Series(labels).dropna().astype(int).unique())
    names = [class_names[i] if 0 <= i < len(class_names) else str(i) for i in labels]
    return labels, names

def evaluate_model(model, X_test, y_test, name, class_names):
    y_pred = model.predict(X_test)
    labels, names = class_names_for_labels(pd.concat([pd.Series(y_test), pd.Series(y_pred)]), class_names)

    print("="*50)
    print(name)
    print("="*50)
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(
        y_test,
        y_pred,
        labels=labels,
        target_names=names,
        zero_division=0,
    ))
    return y_pred, labels, names

y_pred_sys, sys_labels, sys_names = evaluate_model(
    dt_syscall, X_test_sys, y_test_sys, "SYSCALL SCENARIO CLASSIFICATION", SCENARIO_CLASS_NAMES
)
y_pred_net, net_labels, net_names = evaluate_model(
    dt_network, X_test_net, y_test_net, "NETWORK SCENARIO CLASSIFICATION", SCENARIO_CLASS_NAMES
)


In [ ]:
def per_class_support(y_true, y_pred, class_names, name):
    labels, names = class_names_for_labels(pd.concat([pd.Series(y_true), pd.Series(y_pred)]), class_names)
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=names,
        zero_division=0,
        output_dict=True,
    )
    rows = []
    for label, class_name in zip(labels, names):
        metrics = report[class_name]
        rows.append({
            "label": label,
            "class_name": class_name,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1-score"],
            "support": int(metrics["support"]),
        })
    df = pd.DataFrame(rows)
    print(f"{name} per-class metrics")
    display(df)
    return df

syscall_class_metrics = per_class_support(
    y_test_sys, y_pred_sys, SCENARIO_CLASS_NAMES, "Syscall scenario"
)
network_class_metrics = per_class_support(
    y_test_net, y_pred_net, SCENARIO_CLASS_NAMES, "Network scenario"
)


In [ ]:
def normalized_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")
    return np.nan_to_num(cm, nan=0.0)

def plot_reference_confusion_matrices():
    flow_cm = normalized_confusion_matrix(y_test_net, y_pred_net, net_labels)
    host_cm = normalized_confusion_matrix(y_test_sys, y_pred_sys, sys_labels)

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    sns.heatmap(
        flow_cm,
        ax=axes[0],
        annot=True,
        fmt=".3f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        square=True,
        cbar=True,
        xticklabels=net_labels,
        yticklabels=net_labels,
        linewidths=0.5,
        linecolor="white",
    )
    axes[0].set_title("Flow-based Confusion Matrix for Decision Tree", fontsize=13, pad=14)
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    sns.heatmap(
        host_cm,
        ax=axes[1],
        annot=True,
        fmt=".3f",
        cmap="Blues",
        vmin=0,
        vmax=1,
        square=True,
        cbar=True,
        xticklabels=sys_labels,
        yticklabels=sys_labels,
        linewidths=0.5,
        linecolor="white",
    )
    axes[1].set_title("Host-based Confusion Matrix for Decision Tree", fontsize=13, pad=14)
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")

    for ax in axes:
        ax.tick_params(axis="x", rotation=0)
        ax.tick_params(axis="y", rotation=0)

    axes[0].text(
        0.5, -0.16, "(a)",
        transform=axes[0].transAxes,
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold",
        family="serif",
    )
    axes[1].text(
        0.5, -0.16, "(b)",
        transform=axes[1].transAxes,
        ha="center",
        va="center",
        fontsize=18,
        fontweight="bold",
        family="serif",
    )

    plt.tight_layout()
    plt.savefig("models/confusion_matrix_reference_style.png", dpi=200, bbox_inches="tight")
    plt.show()

plot_reference_confusion_matrices()

## Network Flow Distribution


In [ ]:
def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def plot_figure7_style_flow_distribution(network_df):
    in_col = first_existing_column(
        network_df,
        [
            "IN_BYTES",
            "in_bytes",
            "input_bytes",
            "bytes_in",
            "dir_INGRESS_bytes",
            "dir_INGRESS_count",
        ],
    )
    out_col = first_existing_column(
        network_df,
        [
            "OUT_BYTES",
            "out_bytes",
            "output_bytes",
            "bytes_out",
            "dir_EGRESS_bytes",
            "dir_EGRESS_count",
        ],
    )

    if in_col is None or out_col is None:
        if {"dir_INGRESS_count", "dir_EGRESS_count"}.issubset(network_df.columns):
            in_col, out_col = "dir_INGRESS_count", "dir_EGRESS_count"
            title = "Comparison of In/Out Flow Count Distributions (log scale)"
        else:
            raise ValueError("No IN/OUT byte or ingress/egress count columns found.")
    else:
        title = "Comparison of In/Out Bytes Distributions (log scale)"

    plot_df = network_df.copy()
    scenario_col = "scenario_label" if "scenario_label" in plot_df.columns else "label"
    plot_df["class_name"] = plot_df[scenario_col].map(lambda label: SCENARIO_CLASS_NAMES[int(label)])
    plot_df["_in_value"] = plot_df[in_col].clip(lower=0) + 1
    plot_df["_out_value"] = plot_df[out_col].clip(lower=0) + 1

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.scatterplot(
        data=plot_df,
        x="_out_value",
        y="_in_value",
        hue="class_name",
        style="class_name",
        alpha=0.75,
        ax=ax,
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(title, fontsize=13, pad=12)
    ax.set_xlabel(f"OUT ({out_col}) + 1")
    ax.set_ylabel(f"IN ({in_col}) + 1")
    ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.35)
    ax.legend(title="Scenario", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.savefig("models/figure7_flow_distribution_style.png", dpi=200, bbox_inches="tight")
    plt.show()

plot_figure7_style_flow_distribution(network_df)

## Dominant Syscall Sequences


In [ ]:
def format_ngram(row):
    return "-".join(str(int(row[col])) for col in ["n1", "n2", "n3", "n4", "n5"])

def plot_figure9_style_syscall_sequences(syscall_df, top_n=5):
    required = {"n1", "n2", "n3", "n4", "n5"}
    missing = required - set(syscall_df.columns)
    if missing:
        raise ValueError(f"Missing syscall n-gram columns: {sorted(missing)}")

    scenario_col = "scenario_label" if "scenario_label" in syscall_df.columns else "label"
    scenario_labels = sorted(syscall_df[scenario_col].unique())
    if not scenario_labels:
        raise ValueError("No scenario labels found in syscall dataset.")

    ncols = 2
    nrows = int(np.ceil(len(scenario_labels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.5 * nrows))
    axes = np.array(axes).reshape(-1)

    for ax, label in zip(axes, scenario_labels):
        subset = syscall_df[syscall_df[scenario_col] == label].copy()
        subset["sequence"] = subset.apply(format_ngram, axis=1)
        top = subset["sequence"].value_counts().head(top_n).sort_values()

        ax.barh(top.index, top.values, color="#2F6FAE")
        ax.set_title(f"{int(label)} {SCENARIO_CLASS_NAMES[int(label)]}", fontsize=12, pad=10)
        ax.set_xlabel("Frequency")
        ax.set_ylabel("5-gram sequence")
        ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.35)

    for ax in axes[len(scenario_labels):]:
        ax.axis("off")

    fig.suptitle(
        "Frequency Distributions of Dominant System-call Sequences by Scenario",
        fontsize=14,
        y=1.02,
    )
    plt.tight_layout()
    plt.savefig("models/figure9_syscall_sequences_style.png", dpi=200, bbox_inches="tight")
    plt.show()

plot_figure9_style_syscall_sequences(syscall_df)

## Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

def plot_feature_importance(ax, model, columns, title):
    fi = (
        pd.DataFrame({"feature": columns, "importance": model.feature_importances_})
        .sort_values("importance", ascending=False)
        .head(15)
    )
    ax.barh(range(len(fi)), fi["importance"])
    ax.set_yticks(range(len(fi)))
    ax.set_yticklabels(fi["feature"])
    ax.set_title(title)
    ax.invert_yaxis()

plot_feature_importance(axes[0], dt_syscall, X_syscall.columns, "Top 15 Syscall Scenario Features")
plot_feature_importance(axes[1], dt_network, X_network.columns, "Top 15 Network Scenario Features")

plt.tight_layout()
plt.show()


## Visualisasi Decision Tree

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 10))

plot_tree(
    dt_syscall,
    max_depth=3,
    feature_names=X_syscall.columns,
    class_names=[SCENARIO_CLASS_NAMES[i] if 0 <= i < len(SCENARIO_CLASS_NAMES) else str(i) for i in sorted(dt_syscall.classes_)],
    filled=True,
    fontsize=8,
    ax=axes[0],
)
axes[0].set_title("Syscall Scenario Decision Tree (3 levels)", fontsize=14)

plot_tree(
    dt_network,
    max_depth=3,
    feature_names=X_network.columns,
    class_names=[SCENARIO_CLASS_NAMES[i] if 0 <= i < len(SCENARIO_CLASS_NAMES) else str(i) for i in sorted(dt_network.classes_)],
    filled=True,
    fontsize=8,
    ax=axes[1],
)
axes[1].set_title("Network Scenario Decision Tree (3 levels)", fontsize=14)

plt.tight_layout()
plt.show()

## Save Model

In [ ]:
Path("models").mkdir(exist_ok=True)

joblib.dump(dt_syscall, "models/dt_syscall_scenario_model.pkl")
joblib.dump(dt_network, "models/dt_network_scenario_model.pkl")

# Backward-compatible aliases for existing loader code.
joblib.dump(dt_syscall, "models/dt_syscall_model.pkl")
joblib.dump(dt_network, "models/dt_network_model.pkl")

joblib.dump(X_syscall.columns.tolist(), "models/feature_names_syscall.pkl")
joblib.dump(X_network.columns.tolist(), "models/feature_names_network.pkl")
joblib.dump(SCENARIO_CLASS_NAMES, "models/scenario_class_names.pkl")

print("Scenario syscall model: models/dt_syscall_scenario_model.pkl")
print("Scenario network model: models/dt_network_scenario_model.pkl")
print("Feature names and scenario class names saved")
